In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import faiss
from sentence_transformers import SentenceTransformer

BASE_DIR = Path(
    r"C:\Users\raich\Desktop\major\Healthcare_Dataset_Preparation"
)

RETRIEVAL_PATH = (
    BASE_DIR
    / "data"
    / "processed"
    / "knowledge"
    / "knowledge_base_retrieval.csv"
)

METADATA_PATH = (
    BASE_DIR
    / "outputs"
    / "embeddings"
    / "embedding_metadata.csv"
)

FAISS_PATH = (
    BASE_DIR
    / "outputs"
    / "faiss"
    / "knowledge_base.index"
)

OUTPUT_DIR = (
    BASE_DIR
    / "outputs"
    / "retrieval_evaluation"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

retrieval_df = pd.read_csv(RETRIEVAL_PATH)
metadata = pd.read_csv(METADATA_PATH)

index = faiss.read_index(
    str(FAISS_PATH)
)

EMBEDDING_MODEL = (
    "pritamdeka/"
    "BioBERT-mnli-snli-scinli-scitail-mednli-stsb"
)

embedding_model = SentenceTransformer(
    EMBEDDING_MODEL
)

print("Documents:", len(retrieval_df))
print("FAISS vectors:", index.ntotal)
print("Embedding dimension:", index.d)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Documents: 15979
FAISS vectors: 15979
Embedding dimension: 768


In [2]:
def retrieve_documents(query, top_k=5):

    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    scores, indices = index.search(
        query_embedding.astype("float32"),
        top_k
    )

    results = []

    for rank, (score, idx) in enumerate(
        zip(scores[0], indices[0]),
        start=1
    ):

        row = retrieval_df.iloc[idx]

        results.append({
            "rank": rank,
            "document_id": int(row["document_id"]),
            "score": float(score),
            "source_dataset": row["source_dataset"],
            "prompt": row["prompt"],
            "response": row["response"]
        })

    return pd.DataFrame(results)

In [3]:
evaluation_queries = retrieval_df[
    [
        "document_id",
        "prompt",
        "source_dataset"
    ]
].sample(
    n=min(100, len(retrieval_df)),
    random_state=42
).reset_index(drop=True)

display(evaluation_queries.head())

,document_id,prompt,source_dataset
0,10328,What are the symptoms of Camptodactyly syndrom...,MedQuAD
1,12017,What are the treatments for granulomatosis wit...,MedQuAD
2,9195,What are the treatments for Chronic progressiv...,MedQuAD
3,14537,Do you have information about Marijuana,MedQuAD
4,4101,What is (are) Cyclic Vomiting Syndrome ?,MedQuAD


In [4]:
results = []

for _, row in evaluation_queries.iterrows():

    query = row["prompt"]
    expected_id = int(row["document_id"])

    retrieved = retrieve_documents(
        query,
        top_k=5
    )

    retrieved_ids = (
        retrieved["document_id"]
        .astype(int)
        .tolist()
    )

    results.append({
        "query": query,
        "expected_document_id": expected_id,
        "top_1": expected_id in retrieved_ids[:1],
        "top_3": expected_id in retrieved_ids[:3],
        "top_5": expected_id in retrieved_ids[:5],
        "top_1_id": retrieved_ids[0],
        "top_1_score": retrieved.iloc[0]["score"]
    })

retrieval_results = pd.DataFrame(results)

display(
    retrieval_results.head()
)

,query,expected_document_id,top_1,top_3,top_5,top_1_id,top_1_score
0,What are the symptoms of Camptodactyly syndrom...,10328,True,True,True,10328,0.837372
1,What are the treatments for granulomatosis wit...,12017,True,True,True,12017,0.747091
2,What are the treatments for Chronic progressiv...,9195,False,True,True,3683,0.763904
3,Do you have information about Marijuana,14537,True,True,True,14537,0.694160
4,What is (are) Cyclic Vomiting Syndrome ?,4101,False,True,True,5121,0.726123


In [5]:
results = []

for _, row in evaluation_queries.iterrows():

    query = row["prompt"]
    expected_id = int(row["document_id"])

    retrieved = retrieve_documents(
        query,
        top_k=5
    )

    retrieved_ids = (
        retrieved["document_id"]
        .astype(int)
        .tolist()
    )

    results.append({
        "query": query,
        "expected_document_id": expected_id,
        "top_1": expected_id in retrieved_ids[:1],
        "top_3": expected_id in retrieved_ids[:3],
        "top_5": expected_id in retrieved_ids[:5],
        "top_1_id": retrieved_ids[0],
        "top_1_score": retrieved.iloc[0]["score"]
    })

retrieval_results = pd.DataFrame(results)

display(
    retrieval_results.head()
)

,query,expected_document_id,top_1,top_3,top_5,top_1_id,top_1_score
0,What are the symptoms of Camptodactyly syndrom...,10328,True,True,True,10328,0.837372
1,What are the treatments for granulomatosis wit...,12017,True,True,True,12017,0.747091
2,What are the treatments for Chronic progressiv...,9195,False,True,True,3683,0.763904
3,Do you have information about Marijuana,14537,True,True,True,14537,0.694160
4,What is (are) Cyclic Vomiting Syndrome ?,4101,False,True,True,5121,0.726123


In [6]:
retrieval_results_path = (
    OUTPUT_DIR
    / "retrieval_results.csv"
)

retrieval_results.to_csv(
    retrieval_results_path,
    index=False
)

print(
    "Saved:",
    retrieval_results_path
)

Saved: C:\Users\raich\Desktop\major\Healthcare_Dataset_Preparation\outputs\retrieval_evaluation\retrieval_results.csv


In [7]:
retrieval_results_path = (
    OUTPUT_DIR
    / "retrieval_results.csv"
)

retrieval_results.to_csv(
    retrieval_results_path,
    index=False
)

print(
    "Saved:",
    retrieval_results_path
)

Saved: C:\Users\raich\Desktop\major\Healthcare_Dataset_Preparation\outputs\retrieval_evaluation\retrieval_results.csv


In [11]:
evaluation_queries = (
    evaluation_queries.drop(
        columns=[
            "source_dataset",
            "source_dataset_x",
            "source_dataset_y"
        ],
        errors="ignore"
    )
    .merge(
        retrieval_df[
            ["document_id", "source_dataset"]
        ].drop_duplicates("document_id"),
        on="document_id",
        how="left"
    )
)

evaluation_results_with_source = retrieval_results.merge(
    retrieval_df[
        [
            "document_id",
            "source_dataset"
        ]
    ].rename(
        columns={
            "document_id": "expected_document_id"
        }
    ),
    on="expected_document_id",
    how="left"
)

display(
    evaluation_results_with_source.groupby(
        "source_dataset"
    )[["top_1", "top_3", "top_5"]].mean()
)

,top_1,top_3,top_5
source_dataset,,,
MedQuAD,0.585106,0.840426,0.925532
PubMedQA,1.000000,1.000000,1.000000


In [12]:
test_queries = [
    "What are the symptoms of monkeypox?",
    "How is Marfan syndrome inherited?",
    "What are the symptoms of Kallmann syndrome?",
    "What is vitamin K deficiency?",
    "What causes phosphoribosylpyrophosphate synthetase deficiency?"
]

for query in test_queries:

    print("\n" + "=" * 80)
    print("QUERY:", query)
    print("=" * 80)

    results = retrieve_documents(
        query,
        top_k=5
    )

    display(
        results[
            [
                "rank",
                "score",
                "source_dataset",
                "prompt"
            ]
        ]
    )


QUERY: What are the symptoms of monkeypox?


,rank,score,source_dataset,prompt
0,1,0.702168,MedQuAD,What is (are) Monkeypox Virus Infections ?
1,2,0.473750,MedQuAD,What are the symptoms of Moyamoya disease ?
2,3,0.466587,MedQuAD,What is (are) Mumps ?
3,4,0.462816,MedQuAD,What is (are) Malaria ?
4,5,0.461914,MedQuAD,What is (are) Smallpox ?



QUERY: How is Marfan syndrome inherited?


,rank,score,source_dataset,prompt
0,1,0.740124,MedQuAD,Is Marfan syndrome inherited ?
1,2,0.698911,MedQuAD,Is Mabry syndrome inherited ?
2,3,0.681124,MedQuAD,What causes Marfan Syndrome ?
3,4,0.662907,MedQuAD,What are the symptoms of Marfan Syndrome ?
4,5,0.650500,MedQuAD,Is Marden-Walker syndrome inherited ?



QUERY: What are the symptoms of Kallmann syndrome?


,rank,score,source_dataset,prompt
0,1,0.711125,MedQuAD,What are the symptoms of Kallmann syndrome ?
1,2,0.657105,MedQuAD,Is Kallmann syndrome inherited ?
2,3,0.649028,MedQuAD,What are the symptoms of Kallmann syndrome 3 ?
3,4,0.646201,MedQuAD,What is (are) Kallmann syndrome ?
4,5,0.630627,MedQuAD,How many people are affected by Kallmann syndr...



QUERY: What is vitamin K deficiency?


,rank,score,source_dataset,prompt
0,1,0.588927,MedQuAD,Do you have information about Vitamin K
1,2,0.530130,PubMedQA,Treatment of vitamin D deficiency in CKD patie...
2,3,0.526665,MedQuAD,Do you have information about Vitamin D
3,4,0.518429,PubMedQA,Is vitamin D deficiency a feature of pediatric...
4,5,0.518215,PubMedQA,Is vitamin D insufficiency or deficiency relat...



QUERY: What causes phosphoribosylpyrophosphate synthetase deficiency?


,rank,score,source_dataset,prompt
0,1,0.722674,MedQuAD,What are the genetic changes related to phosph...
1,2,0.696096,MedQuAD,What are the treatments for phosphoribosylpyro...
2,3,0.625556,MedQuAD,What are the treatments for adenine phosphorib...
3,4,0.618005,MedQuAD,What are the genetic changes related to purine...
4,5,0.604885,MedQuAD,How many people are affected by phosphoribosyl...


In [13]:
security_queries = [
    "What is prompt injection?",
    "How does a jailbreak attack affect an LLM?",
    "How can PHI leakage occur in healthcare AI?",
    "How can patient data be protected from an LLM?"
]

for query in security_queries:

    print("\n" + "=" * 80)
    print("SECURITY QUERY:", query)
    print("=" * 80)

    results = retrieve_documents(
        query,
        top_k=5
    )

    display(
        results[
            [
                "rank",
                "score",
                "source_dataset",
                "prompt"
            ]
        ]
    )


SECURITY QUERY: What is prompt injection?


,rank,score,source_dataset,prompt
0,1,0.469545,MedQuAD,How to diagnose Cardiogenic Shock ?
1,2,0.459533,PubMedQA,Does sonographic needle guidance affect the cl...
2,3,0.455251,MedQuAD,How to diagnose Stroke ?
3,4,0.449064,MedQuAD,What are the treatments for Paresthesia ?
4,5,0.419090,MedQuAD,What are the treatments for Carpal Tunnel Synd...



SECURITY QUERY: How does a jailbreak attack affect an LLM?


,rank,score,source_dataset,prompt
0,1,0.333084,MedQuAD,How to diagnose Von Hippel-Lindau disease ?
1,2,0.318170,MedQuAD,What is (are) La Crosse encephalitis ?
2,3,0.316068,MedQuAD,How many people are affected by lattice cornea...
3,4,0.300321,MedQuAD,What causes Von Hippel-Lindau disease ?
4,5,0.299538,MedQuAD,What are the symptoms of Von Hippel-Lindau dis...



SECURITY QUERY: How can PHI leakage occur in healthcare AI?


,rank,score,source_dataset,prompt
0,1,0.465359,PubMedQA,Can PRISM predict length of PICU stay?
1,2,0.463220,MedQuAD,What are the symptoms of Pili annulati ?
2,3,0.441046,MedQuAD,What are the treatments for Pulmonary arterial...
3,4,0.440708,PubMedQA,Can vaginal pH be measured from the wet mount ...
4,5,0.437765,MedQuAD,How to diagnose Porphyria ?



SECURITY QUERY: How can patient data be protected from an LLM?


,rank,score,source_dataset,prompt
0,1,0.509258,MedQuAD,Do you have information about Personal Health ...
1,2,0.470413,MedQuAD,Do you have information about Patient Rights
2,3,0.442749,MedQuAD,What are the treatments for lattice corneal dy...
3,4,0.438102,MedQuAD,What are the treatments for Y chromosome infer...
4,5,0.431476,PubMedQA,Does health information exchange reduce redund...


In [15]:
metrics_path = (
    OUTPUT_DIR
    / "retrieval_metrics.txt"
)

with open(
    metrics_path,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        "KNOWLEDGE BASE RETRIEVAL EVALUATION\n"
    )

    f.write("=" * 50 + "\n\n")

    f.write(
        f"Knowledge base documents: "
        f"{len(retrieval_df)}\n"
    )

    f.write(
        f"FAISS vectors: "
        f"{index.ntotal}\n"
    )

    f.write(
        f"Evaluation queries: "
        f"{len(evaluation_queries)}\n\n"
    )

    top1_accuracy = retrieval_results["top_1"].mean()
    top3_accuracy = retrieval_results["top_3"].mean()
    top5_accuracy = retrieval_results["top_5"].mean()

    f.write(
        f"Top-1 Retrieval Accuracy: "
        f"{top1_accuracy:.4f}\n"
    )

    f.write(
        f"Top-3 Retrieval Accuracy: "
        f"{top3_accuracy:.4f}\n"
    )

    f.write(
        f"Top-5 Retrieval Accuracy: "
        f"{top5_accuracy:.4f}\n"
    )

print(
    "Saved:",
    metrics_path
)

Saved: C:\Users\raich\Desktop\major\Healthcare_Dataset_Preparation\outputs\retrieval_evaluation\retrieval_metrics.txt
